# Ray Train: Distributed training on Ray

While serving models is a great usecase of Ray for ML Engineers or ML Ops folks, data scientists can also benefit from Ray by distributing training itself.

Please note that a strange observation: While Scikit-Learn is the main library for classical ML in Python, Ray seems to go out of its way to not mention it very much on its site. There are examples of distributed deep learning libraries, such as PyTorch, and classical ML libraries like XGBoost. However, scikit-learn is mentioned only in the dark corners of the manual. Despite that, since this class assumes familiarity with Scikit-Learn, we will show how to distributed SKLearn code.

In [ ]:
import time
import pandas as pd
import joblib

import ray
from ray.util.joblib import register_ray

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import fetch_openml

#### Setup data

In [ ]:
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True, parser='pandas')
# Simple preprocessing for the demo
X = X[['pclass', 'sibsp', 'parch']].fillna(0) 

#### Define a model
Note that this model must support the `n_jobs` attribute. This attribute uses the `joblib` library to distribute execution across multiple cpu cores. 

In [ ]:
# Notice the large number of n_estimators!
model = RandomForestClassifier(n_estimators=5_000, max_depth=100, random_state=42)

#### Single process execution

In [ ]:
%%time
scores = cross_val_score(model, X, y, cv=15, n_jobs=1)
scores.mean()

#### Using Joblib (internally) for local multiprocessing

In [ ]:
%%time
scores = cross_val_score(model, X, y, cv=15, n_jobs=-1) # Notice n_jobs being set to -1, meaning "use all cpus"
scores.mean()

#### Ray backed joblib for (simulated) distributed processing

In [ ]:
if not ray.is_initialized():
    ray.init(runtime_env={"env_vars": {"PYTHONWARNINGS": "ignore"}})
    #ray.init(address="ray://localhost:10001", ignore_reinit_error=True) # Remote/docker cluster

#print(ray.cluster_resources())

In [ ]:
# Now register with joblib
register_ray()

In [ ]:
%%time
with joblib.parallel_backend('ray'):
    scores = cross_val_score(model, X, y, cv=15, n_jobs=-1)
    
scores.mean()

### The algorithm must be parallelizable
Ideal algorithms which benefit from distribution are the ones where sub-jobs can operate on their own, with as little coordination as possible. For example, cross validtion will essentially need run `.fit()` methods independently.

#### Embarrasingly parallel
"[Embarrasingly parallel](https://en.wikipedia.org/wiki/Embarrassingly_parallel)" is an actual technical term! Take the example of training the same model with different parameters. This is the perfect problem for distributing across a cluster. There are almost no dependencies among the runs! We are not always so lucky.

#### Interdependent jobs
Imagine you have a thousand celestial bodies, perhaps asteroids and you are trying to determine how they interact based on gravity. Perhaps you are trying to simulate a massive game of billairds with a hundred balls. Maybe you need to extrapolate a time series or a language model, one element at time. Unfortunately these problems are not easy to distribute. There is too much dependency among the calculations for them to operate in isolation.

### The benefit of parallelization may not show up until the scale of the problem is increased
Recall how to measure performance differences: there is always some overhead involved in parallelization. If you are fitting two logistic regression model that take a tenth of a second to train, the cost of sending them across the cluster and time it takes to do the coordinationand book keeping may be higher than just training the two models on the same machine! However, if you have 10,000 models to fit, perhaps the book keeping cost is amortized across all the models and is reduced to almost nothing!

In performance, scale matters!

# Ray Tune: Distributed hyper parameter tuning 

Ray Tune provides a library specifically for distributed hyper parameter tuning. It works with many pre-existing libraries, such as HyperOpt, BayesOpt, etc. However, it doesn't not have a native implementation for Scikit-Learn. However, here is a usable implementation

In [ ]:
from ray import tune
from ray.tune import Tuner, TuneConfig

Ray Tune requires an "Objective" function which instantiates an ML model with specific parameters. This function will be called repeatedly and different parameters will be provided:

In [ ]:
def train_sklearn(config):
    # This is standard scikit-learn inside the worker
    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"]
    )
    
    # We use cross-validation and take the mean score
    scores = cross_val_score(model, X, y, cv=3)
    mean_accuracy = scores.mean()
    
    # REPORT the result back to the Ray head node
    ray.tune.report({"mean_accuracy": mean_accuracy})


Create parameter values to try:

In [ ]:
param_space = {
    "n_estimators": tune.grid_search([10, 50, 100]),
    "max_depth": tune.grid_search([5, 10, None])
}

Run the `Tuner` class

In [ ]:
%%time
tuner = Tuner(
    train_sklearn,
    param_space=param_space,
    tune_config=TuneConfig(metric="mean_accuracy", mode="max")
)

results = tuner.fit()

Print the results of the best model:

In [ ]:
best_result = results.get_best_result()

print(f"Best Config: {best_result.config}")
print(f"Best Accuracy: {best_result.metrics['mean_accuracy']:.4f}")

In [ ]:
# Cleanup ray
ray.shutdown()

### Additional features
Note that this is only the most introductory exampele of Ray Tune. As this lecture develops, additonal examples for early stopping and the scheduler will be included